# PMI Mismatch Guardrails Demonstration

This notebook demonstrates the guardrails implemented to prevent cryptic `MPICH_jll` aborts when running on Slurm clusters with mismatched PMI versions (e.g., PMIx vs PMI2).

We will show the behavior of `FluxMPIExt.jl`'s initialization wrapper and the standalone health check script.

In [ ]:
import Pkg
Pkg.activate("..") # Activate the ddp-flux-thesis project
using Flux, MPI

println("Loaded Flux and MPI.")

## Guardrail 1: Initialization Wrapper

By default, `MPICH_jll` expects the **PMI2** wire protocol. If you launch with `srun --mpi=pmix` (or if it's the cluster default), Slurm sets the `PMIX_RANK` environment variable.

Before reaching `MPI.Init()`, the initialization wrapper checks the environment and gracefully throws an informative error instead of a hard native abort.

In [ ]:
# Simulate a PMIx Slurm environment
ENV["PMIX_RANK"] = "1"

try
    Flux.DistributedUtils.initialize(Flux.MPIBackend)
catch e
    println("✅ Caught expected error:\n")
    showerror(stdout, e)
    println()
end

delete!(ENV, "PMIX_RANK") # Cleanup

A similar issue occurs if the user launches with OpenMPI (`mpirun`), but the Julia environment loads `MPICH_jll`. The wrapper detects `OMPI_COMM_WORLD_RANK` and catches this too.

In [ ]:
# Simulate an OpenMPI environment
ENV["OMPI_COMM_WORLD_RANK"] = "0"

try
    Flux.DistributedUtils.initialize(Flux.MPIBackend)
catch e
    println("✅ Caught expected error:\n")
    showerror(stdout, e)
    println()
end

delete!(ENV, "OMPI_COMM_WORLD_RANK") # Cleanup

If we are inside a Slurm allocation (`SLURM_JOB_ID` is present), but no explicit `PMI2_*` keys are found, the wrapper emits a warning just in case `MPI.Init()` hangs or fails, guiding the user to the correct launcher argument.

*(Note: Since this doesn't error, `MPI.Init()` will actually be called below, initializing MPI for the rest of the notebook.)*

In [ ]:
# Simulate a generic Slurm environment without PMI2 variables
ENV["SLURM_JOB_ID"] = "99999"

println("Calling initialize... (Expect a warning to be logged)")
Flux.DistributedUtils.initialize(Flux.MPIBackend)

println("\nMPI.Initialized() is now: ", MPI.Initialized())

delete!(ENV, "SLURM_JOB_ID") # Cleanup

## Guardrail 3: Health Check Script

We also added a standalone health check script that isolates initialization errors from training code. We invoke it via `make health-check`. It prints the environment variables and the exact MPI version being used.

In [ ]:
cd("..") do
    # We use `make health-check` which defaults to 2 processes (local mpiexecjl or remote srun)
    println("Running: make health-check\n")
    output = read(`make health-check`, String)
    
    # Print the output from Rank 0 
    println(output)
end